# 🍽️ Yemek Asistanı
LangGraph + Groq + Gradio + SQLite

In [ ]:
# 1. Kurulum
!pip install -q langchain-groq langgraph gradio

In [ ]:
# 2. API Key — Colab Secrets (sol menüde 🔑 simgesi) → GROQ_API_KEY ekle
from google.colab import userdata
import os
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
print('✅ API key yüklendi')

In [ ]:
# 3. Kodu indir ve çalıştır
import requests

# Dosyayı bu hücreye yapıştır ya da aşağıdaki gibi direkt tanımla:
code = '''
import os, re, sqlite3, json
import gradio as gr
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
MODEL_NAME   = "gemma2-9b-it"
DB_PATH      = "favorites.db"

SYSTEM_PROMPT = """Sen yardımsever bir Türk yemek asistanısın.

VARSAYILAN TERCİHLER:
- Kişi sayısı: 3
- Öğün: Akşam yemeği
- Tarz: Ev yemeği (pratik, hafif, fit)
- Tercihler: Düşük kalorili, proteinli, glutensiz, sebze ağırlıklı, tavuk veya dana eti, fırın veya tencere yemekleri

KESİNLİKLE ÖNERİLMEYECEKLER:
- Kuzu eti, Uzakdoğu mutfağı, Noodle, Soya sosu, Teriyaki, Sushi, Ramen, Wok tarzı yemekler

YEMEK ÖNERİSİ KURALLARI:
Ne yiyelim, yemek öner, akşam yemeği, bugün ne yesek gibi sorular geldiğinde MUTLAKA aşağıdaki formatta JSON döndür.
Başka hiçbir şey yazma, sadece JSON:

{
  \"type\": \"meal_suggestion\",
  \"categories\": {
    \"tavuk\": [\"Yemek 1\", \"Yemek 2\", \"Yemek 3\"],
    \"dana_et\": [\"Yemek 1\", \"Yemek 2\", \"Yemek 3\"],
    \"sebze\": [\"Yemek 1\", \"Yemek 2\", \"Yemek 3\"],
    \"bakliyat\": [\"Yemek 1\", \"Yemek 2\", \"Yemek 3\"],
    \"fit_salata\": [\"Yemek 1\", \"Yemek 2\", \"Yemek 3\"],
    \"balik\": [\"Yemek 1\", \"Yemek 2\", \"Yemek 3\"]
  }
}

Her kategoride tam olarak 3 öneri ver. Türkçe yemek isimleri kullan.
Yemekle ilgili OLMAYAN sorularda normal, yardımsever bir asistan olarak cevap ver (JSON kullanma).
"""

CATEGORY_LABELS = {
    "tavuk":      "🍗 Tavuk Ağırlıklı",
    "dana_et":    "🥩 Dana / Et Ağırlıklı",
    "sebze":      "🥦 Sebze Ağırlıklı",
    "bakliyat":   "🫘 Bakliyat / Proteinli",
    "fit_salata": "🥗 Fit & Hafif / Salatalar",
    "balik":      "🐟 Balık Ağırlıklı",
}

def init_db():
    conn = sqlite3.connect(DB_PATH)
    conn.execute("CREATE TABLE IF NOT EXISTS favorites (id INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT UNIQUE NOT NULL)")
    conn.commit(); conn.close()

def add_favorite(name):
    conn = sqlite3.connect(DB_PATH)
    conn.execute("INSERT OR IGNORE INTO favorites (name) VALUES (?)", (name,))
    conn.commit(); conn.close()

def get_favorites():
    conn = sqlite3.connect(DB_PATH)
    rows = conn.execute("SELECT name FROM favorites ORDER BY id DESC").fetchall()
    conn.close()
    return [r[0] for r in rows]

def remove_favorite(name):
    conn = sqlite3.connect(DB_PATH)
    conn.execute("DELETE FROM favorites WHERE name = ?", (name,))
    conn.commit(); conn.close()

class AgentState(TypedDict):
    messages: list
    response_text: str
    meal_data: dict

def call_llm(state):
    llm = ChatGroq(api_key=GROQ_API_KEY, model=MODEL_NAME, temperature=0.7, max_tokens=1024)
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    response = llm.invoke(messages)
    raw = response.content.strip()
    meal_data = None
    try:
        clean = re.sub(r"```json|```", "", raw).strip()
        parsed = json.loads(clean)
        if parsed.get("type") == "meal_suggestion":
            meal_data = parsed["categories"]
            raw = "__MEAL_SUGGESTION__"
    except: pass
    return {**state, "response_text": raw, "meal_data": meal_data}

def build_graph():
    g = StateGraph(AgentState)
    g.add_node("llm", call_llm)
    g.set_entry_point("llm")
    g.add_edge("llm", END)
    return g.compile()

graph = build_graph()

def build_meal_html(categories):
    html = "<div style=\'display:flex;flex-direction:column;gap:16px;padding:8px 0\'>"
    for key, label in CATEGORY_LABELS.items():
        items = categories.get(key, [])
        if not items: continue
        html += f"<div style=\'background:#1e1e2e;border:1px solid #313244;border-radius:12px;padding:16px\'>"
        html += f"<div style=\'font-weight:700;font-size:15px;margin-bottom:10px;color:#cdd6f4\'>{label}</div>"
        html += "<div style=\'display:flex;flex-direction:column;gap:8px\'>"
        for item in items:
            safe = item.replace(\'\"\', \'&quot;\').replace("\'"  , \'&#39;\')
            html += f"""<div style=\'display:flex;justify-content:space-between;align-items:center;background:#181825;border-radius:8px;padding:8px 12px\'>
              <span style=\'color:#cdd6f4;font-size:14px\'>{item}</span>
              <button onclick=\'addFavorite("{safe}")\' style=\'background:none;border:none;cursor:pointer;font-size:18px;padding:2px 4px;border-radius:4px\' title=\'Favorilere ekle\'>⭐</button>
            </div>"""
        html += "</div></div>"
    html += "</div>"
    return html

def favorites_html(favs):
    if not favs: return "<p style=\'color:#6c7086;padding:16px\'>Henüz favori yemek eklenmedi.</p>"
    html = "<div style=\'display:flex;flex-direction:column;gap:8px;padding:8px 0\'>"
    for fav in favs:
        safe = fav.replace(\'\"\', \'&quot;\').replace("\'"  , \'&#39;\')
        html += f"""<div style=\'display:flex;justify-content:space-between;align-items:center;background:#1e1e2e;border:1px solid #313244;border-radius:8px;padding:10px 14px\'>
          <span style=\'color:#cdd6f4\'>⭐ {fav}</span>
          <button onclick=\'removeFavorite("{safe}")\' style=\'background:none;border:none;cursor:pointer;color:#f38ba8;font-size:14px;font-weight:600;padding:2px 6px\' title=\'Kaldır\'>✕</button>
        </div>"""
    html += "</div>"
    return html

def chat(user_msg, history, fav_state):
    lc_messages = []
    for msg in history:
        if msg["role"] == "user": lc_messages.append(HumanMessage(content=msg["content"]))
        elif msg["content"] != "__MEAL_SUGGESTION__": lc_messages.append(AIMessage(content=msg["content"]))
    lc_messages.append(HumanMessage(content=user_msg))
    result = graph.invoke({"messages": lc_messages, "response_text": "", "meal_data": None})
    history.append({"role": "user", "content": user_msg})
    if result["meal_data"]:
        history.append({"role": "assistant", "content": build_meal_html(result["meal_data"])})
    else:
        history.append({"role": "assistant", "content": result["response_text"]})
    return history, history, fav_state, favorites_html(fav_state), ""

def handle_add_favorite(name, fav_state):
    if name and name not in fav_state:
        fav_state = [name] + fav_state
        add_favorite(name)
    return fav_state, favorites_html(fav_state), f"✅ {name} eklendi"

def handle_remove_favorite(name, fav_state):
    fav_state = [f for f in fav_state if f != name]
    remove_favorite(name)
    return fav_state, favorites_html(fav_state), f"🗑️ {name} kaldırıldı"

CSS = \"\"\"
@import url(\'https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;600;700&display=swap\');
* { box-sizing: border-box; }
body, .gradio-container { background: #11111b !important; font-family: \'Plus Jakarta Sans\', sans-serif !important; color: #cdd6f4 !important; }
footer { display: none !important; }
.message.user { background: #313244 !important; color: #cdd6f4 !important; border-radius: 12px !important; }
.message.bot  { background: #1e1e2e !important; color: #cdd6f4 !important; border-radius: 12px !important; }
#send-btn { background: linear-gradient(135deg, #89b4fa, #b4befe) !important; color: #11111b !important; font-weight: 700 !important; border: none !important; border-radius: 10px !important; }
#input-box textarea { background: #1e1e2e !important; color: #cdd6f4 !important; border: 1px solid #313244 !important; border-radius: 10px !important; }
.title-bar { text-align:center; padding:20px 0 8px; font-size:26px; font-weight:700; color:#89b4fa; }
.subtitle  { text-align:center; color:#6c7086; font-size:13px; margin-bottom:12px; }
\"\"\"

JS_BRIDGE = \"\"\"
<script>
function addFavorite(name) {
    const inp = document.getElementById(\'fav-add-input\');
    if (inp) {
        const n = inp.querySelector(\'input\') || inp.querySelector(\'textarea\');
        if (n) { Object.getOwnPropertyDescriptor(window.HTMLInputElement.prototype,\'value\').set.call(n,name); n.dispatchEvent(new Event(\'input\',{bubbles:true})); }
    }
    setTimeout(()=>{ const b=document.getElementById(\'fav-add-btn\'); if(b)b.click(); },100);
}
function removeFavorite(name) {
    const inp = document.getElementById(\'fav-remove-input\');
    if (inp) {
        const n = inp.querySelector(\'input\') || inp.querySelector(\'textarea\');
        if (n) { Object.getOwnPropertyDescriptor(window.HTMLInputElement.prototype,\'value\').set.call(n,name); n.dispatchEvent(new Event(\'input\',{bubbles:true})); }
    }
    setTimeout(()=>{ const b=document.getElementById(\'fav-remove-btn\'); if(b)b.click(); },100);
}
</script>
\"\"\"

init_db()
initial_favs = get_favorites()

with gr.Blocks(css=CSS, title="Yemek Asistanı") as demo:
    history_state = gr.State([])
    fav_state     = gr.State(initial_favs)
    gr.HTML(JS_BRIDGE)
    gr.HTML("<div class=\'title-bar\'>🍽️ Yemek Asistanı</div>")
    gr.HTML("<div class=\'subtitle\'>Ne yesek? Sor bakalım.</div>")
    with gr.Tabs():
        with gr.Tab("💬 Sohbet"):
            chatbot = gr.Chatbot(label="", height=480, type="messages", render_markdown=False, sanitize_html=False, bubble_full_width=False)
            status_box = gr.HTML()
            with gr.Row():
                user_input = gr.Textbox(placeholder="Ne yiyelim bugün? ya da istediğin bir şey sor...", show_label=False, elem_id="input-box", scale=5)
                send_btn = gr.Button("Gönder", elem_id="send-btn", scale=1)
        with gr.Tab("⭐ Favoriler"):
            gr.HTML("<div style=\'font-weight:700;font-size:16px;color:#89b4fa;margin-bottom:12px\'>Favori Yemeklerim</div>")
            fav_display = gr.HTML(favorites_html(initial_favs))
            fav_status  = gr.HTML()
    fav_add_input    = gr.Textbox(visible=False, elem_id="fav-add-input")
    fav_add_btn      = gr.Button(visible=False, elem_id="fav-add-btn")
    fav_remove_input = gr.Textbox(visible=False, elem_id="fav-remove-input")
    fav_remove_btn   = gr.Button(visible=False, elem_id="fav-remove-btn")
    def submit(msg, hist, favs):
        if not msg.strip(): return hist, hist, favs, favorites_html(favs), ""
        return chat(msg, hist, favs)
    send_btn.click(fn=submit, inputs=[user_input,history_state,fav_state], outputs=[history_state,chatbot,fav_state,fav_display,status_box]).then(lambda:"",outputs=user_input)
    user_input.submit(fn=submit, inputs=[user_input,history_state,fav_state], outputs=[history_state,chatbot,fav_state,fav_display,status_box]).then(lambda:"",outputs=user_input)
    fav_add_btn.click(fn=handle_add_favorite, inputs=[fav_add_input,fav_state], outputs=[fav_state,fav_display,fav_status]).then(lambda:"",outputs=fav_add_input)
    fav_remove_btn.click(fn=handle_remove_favorite, inputs=[fav_remove_input,fav_state], outputs=[fav_state,fav_display,fav_status]).then(lambda:"",outputs=fav_remove_input)

demo.launch(share=True, debug=False)
'''

with open('food_assistant.py', 'w', encoding='utf-8') as f:
    f.write(code)

print('✅ Kod yazıldı')

In [ ]:
# 4. Çalıştır
%run food_assistant.py